## Step 1: Import Required Libraries
**What this does:** Imports data handling (`pandas`, `numpy`), safe parsing (`ast`), clustering (`KMeans`), and drift metric (`jensenshannon`).
**How it works:** These imports prepare all core utilities needed for training clusters and comparing narrative distributions.
**Visual purpose:** Keeps the workflow readable by clearly setting up dependencies before any data operations.

In [42]:
import pandas as pd
import numpy as np
import ast

from sklearn.cluster import KMeans
from scipy.spatial.distance import jensenshannon
from pathlib import Path    

## Step 2: Define Embedding Parser
**What this does:** Creates `parse_embedding(x)` to convert stored embeddings into clean NumPy vectors.
**How it works:** It safely parses string-form embeddings, converts to array format, and flattens nested shapes into 1D vectors.
**Visual purpose:** Standardizes vector shape so later matrix creation and model fitting are consistent and easy to inspect.

In [43]:
def parse_embedding(x):

    if isinstance(x, str):
        emb = ast.literal_eval(x)
    else:
        emb = x

    emb = np.array(emb)

    if emb.ndim > 1:
        emb = emb.flatten()

    return emb

## Step 3: Configure Single User Input
**What this does:** Sets the single input file path and topic list for the full pipeline.
**How it works:** Defines `USER_CSV_PATH`, `TOPICS`, and core constants used across inference, clustering, and drift steps.
**Visual purpose:** Centralizes runtime configuration so path and schema changes are easy to handle.

In [44]:
from pathlib import Path

TOPICS = ["War", "Health", "Technology", "Climate", "Economics"]
NUM_CLUSTERS = 5

candidate_paths = [
    Path("K_Means_Drift/user_article2.csv"),
    Path("user_article2.csv"),
    Path.cwd() / "K_Means_Drift" / "user_article2.csv",
    Path.cwd() / "user_article2.csv",
    Path.cwd().parent / "K_Means_Drift" / "user_article2.csv"
]

resolved_path = next((p for p in candidate_paths if p.exists()), None)
if resolved_path is None:
    raise FileNotFoundError(
        "Could not find user_article2.csv. Tried: " + ", ".join(str(p) for p in candidate_paths)
    )

USER_CSV_PATH = str(resolved_path)
user_input_df = pd.read_csv(USER_CSV_PATH)

# Handle trailing comma schema variants (e.g., extra unnamed empty column).
user_input_df = user_input_df.loc[:, ~user_input_df.columns.str.contains(r"^Unnamed")].copy()

print("Resolved input path:", USER_CSV_PATH)
print("User input rows:", len(user_input_df))
print("Columns:", list(user_input_df.columns))
user_input_df.head()

Resolved input path: user_article2.csv
User input rows: 5
Columns: ['date', 'article']


,date,article
0,15/02/2022,Military tensions between Russia and Ukraine c...
1,25/02/2022,Russian forces launched a large-scale military...
2,15/03/2022,As the fighting in Ukraine entered its third w...
3,01/04/2022,"After weeks of intense fighting, Russian and U..."
4,20/04/2022,Global attention has increasingly shifted towa...


## Step 4: Validate Input Schema
**What this does:** Checks that the input file matches the new expected format.
**How it works:** Verifies required columns (`date`, `article`) and reports missing columns early.
**Visual purpose:** Prevents downstream failures by validating format immediately after load.

In [45]:
required_cols = {"date", "article"}
missing_cols = required_cols - set(user_input_df.columns)

if missing_cols:
    raise ValueError(f"Missing required columns in user input: {missing_cols}")

print("Input format valid. Required columns present:", required_cols)

Input format valid. Required columns present: {'date', 'article'}


## Step 5: Initialize Pipeline Settings
**What this does:** Prepares shared config values for the user-inference flow.
**How it works:** Uses a single config dictionary consumed by context building, SBERT embeddings, and topic filtering.
**Visual purpose:** Keeps all tunable settings visible in one place for quick experiments.

In [46]:
INFERENCE_CONFIG = {
    "topics": TOPICS,
    "context_window": 3,
    "inference_batch_size": 16,
    "topic_threshold": 0.27,
    "embedding_dim": 768
}

print("Topics:", TOPICS)
print("Num clusters:", NUM_CLUSTERS)
print("Inference config:", INFERENCE_CONFIG)

Topics: ['War', 'Health', 'Technology', 'Climate', 'Economics']
Num clusters: 5
Inference config: {'topics': ['War', 'Health', 'Technology', 'Climate', 'Economics'], 'context_window': 3, 'inference_batch_size': 16, 'topic_threshold': 0.27, 'embedding_dim': 768}


## Step 6: Notes on Training Source
**What this does:** Clarifies the updated training strategy.
**How it works:** Topic-wise K-Means models are now trained directly on filtered user sentence embeddings (after SBERT + soft labels), not on Climate.csv.
**Visual purpose:** Aligns model training with the new single-input workflow.

In [47]:
print("Model training will run after Step 10 using filtered user-topic embeddings.")

Model training will run after Step 10 using filtered user-topic embeddings.


## Step 7: Deferred Topic-wise K-Means Training
**What this does:** Reserves model fitting for post-labeling stage.
**How it works:** After `filter_user_topic_sentences`, each topic gets its own embedding matrix and K-Means model if enough rows exist.
**Visual purpose:** Ensures clustering is performed on the correct topic-filtered user data.

In [48]:
print("Deferred: topic_models will be created in Step 11.")

Deferred: topic_models will be created in Step 11.


## Step 8: Run User Inference Input Pipeline (Steps 1-3)
**What this does:** Loads `user_article2.csv`, splits articles into sentences, builds context windows, and generates contextual SBERT embeddings.
**How it works:** Implements `split_articles_into_sentences -> build_context_texts -> generate_contextual_sbert_embeddings` from the TCL inference path.
**Visual purpose:** Converts raw user articles into sentence-level embedding records ready for topic scoring.

In [49]:
from sentence_transformers import SentenceTransformer

def split_articles_into_sentences(input_dataframe):
    import re

    sentence_rows = []
    required_cols = {"date", "article"}
    missing = required_cols - set(input_dataframe.columns)
    if missing:
        raise ValueError(f"Input CSV must contain columns: {required_cols}. Missing: {missing}")

    for article_idx, row in input_dataframe.reset_index(drop=True).iterrows():
        article_id = row.get("article_id", f"article_{article_idx}")
        date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)
        if pd.isna(date_value):
            continue

        text = str(row["article"]).strip()
        if not text or text.lower() == "nan":
            continue

        sentence_list = [
            s.strip()
            for s in re.split(r"(?<=[.!?])\s+", text)
            if s and s.strip()
        ]

        for sentence_order, sentence_text in enumerate(sentence_list):
            sentence_id = f"{article_id}_s{sentence_order}"
            sentence_rows.append({
                "date": date_value.normalize(),
                "article_id": str(article_id),
                "sentence_id": sentence_id,
                "sentence_text": sentence_text,
                "sentence_order": int(sentence_order)
            })

    sentence_dataframe = pd.DataFrame(sentence_rows)
    if sentence_dataframe.empty:
        return pd.DataFrame(columns=["date", "article_id", "sentence_id", "sentence_text", "sentence_order"])
    return sentence_dataframe[["date", "article_id", "sentence_id", "sentence_text", "sentence_order"]]

def build_context_texts(sentence_dataframe, context_window):
    if context_window not in (3, 5):
        raise ValueError("context_window must be 3 or 5")

    radius = context_window // 2
    sentence_dataframe = sentence_dataframe.sort_values(["article_id", "sentence_order"]).reset_index(drop=True).copy()
    context_texts = [""] * len(sentence_dataframe)

    for _, group in sentence_dataframe.groupby("article_id", sort=False):
        indices = group.index.tolist()
        sentences = group["sentence_text"].tolist()

        for local_idx, global_idx in enumerate(indices):
            left = max(0, local_idx - radius)
            right = min(len(sentences), local_idx + radius + 1)
            context_texts[global_idx] = " ".join(sentences[left:right])

    sentence_dataframe["context_text"] = context_texts
    return sentence_dataframe

def generate_contextual_sbert_embeddings(sentence_dataframe, config, sbert_model_name="all-mpnet-base-v2"):
    if sentence_dataframe.empty:
        sentence_dataframe["sentence_embeddings"] = []
        return sentence_dataframe

    if int(config["embedding_dim"]) != 768:
        raise ValueError("Inference requires config['embedding_dim'] == 768")

    model_sbert = SentenceTransformer(sbert_model_name, device="cpu")
    encoded = model_sbert.encode(
        sentence_dataframe["context_text"].tolist(),
        batch_size=int(config["inference_batch_size"]),
        show_progress_bar=False,
        convert_to_numpy=True
    )
    encoded = np.asarray(encoded, dtype=np.float32)

    if encoded.shape[1] != config["embedding_dim"]:
        raise ValueError(
            f"SBERT output dim {encoded.shape[1]} does not match config['embedding_dim']={config['embedding_dim']}"
        )

    sentence_dataframe = sentence_dataframe.copy()
    sentence_dataframe["sentence_embeddings"] = [vec.astype(np.float32) for vec in encoded]
    return sentence_dataframe

print("Call order:")
print("1. split_articles_into_sentences")
print("2. build_context_texts")
print("3. generate_contextual_sbert_embeddings")
print("4. soft_topic_label_sentences")
print("5. filter_user_topic_sentences")

sentence_df = split_articles_into_sentences(user_input_df)
sentence_df = build_context_texts(sentence_df, int(INFERENCE_CONFIG["context_window"]))
sentence_df = generate_contextual_sbert_embeddings(sentence_df, INFERENCE_CONFIG)

print("User articles:", len(user_input_df))
print("Sentence rows:", len(sentence_df))
sentence_df[["date", "article_id", "sentence_id", "sentence_text"]].head()

Call order:
1. split_articles_into_sentences
2. build_context_texts
3. generate_contextual_sbert_embeddings
4. soft_topic_label_sentences
5. filter_user_topic_sentences


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3075.61it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


User articles: 5
Sentence rows: 28


,date,article_id,sentence_id,sentence_text
0,2022-02-15,article_0,article_0_s0,Military tensions between Russia and Ukraine c...
1,2022-02-15,article_0,article_0_s1,Western intelligence agencies reported large m...
2,2022-02-15,article_0,article_0_s2,Ukrainian officials warned that the buildup co...
3,2022-02-15,article_0,article_0_s3,NATO leaders urged Russia to de-escalate the s...
4,2022-02-15,article_0,article_0_s4,"Meanwhile, residents in eastern Ukraine report..."


## Step 9: Save Topic Models (After Training)
**What this does:** Saves topic-wise K-Means models after they are created from user-topic embeddings.
**How it works:** Writes `topic_models` dictionary if available.
**Visual purpose:** Allows reproducible reruns without retraining when models are already built.

In [50]:
import pickle

if "topic_models" in globals() and len(topic_models) > 0:
    pickle.dump(topic_models, open("kmeans_narrative_models.pkl", "wb"))
    print("Topic-wise models saved")
else:
    print("No topic models to save yet. Run Step 11 first.")

Topic-wise models saved


## Step 10: Soft Topic Labeling + Topic Filtering (Steps 4-5)
**What this does:** Computes topic affinities for each sentence and selects sentence sets per topic.
**How it works:** Loads ideal-topic embeddings from `topic_embeddings.json`, compares sentence embeddings to these prototypes, assigns a dominant topic, and includes a sentence in every topic where affinity is at least `0.35`.
**Visual purpose:** Enables multi-label topic membership using your ideal-topic vectors as the soft-label reference.

In [51]:
import json
from pathlib import Path

def load_topic_embedding_prototypes_from_json(topics, config, json_path="K_Means_Drift/topic_embeddings.json"):
    candidate_paths = [
        Path(json_path),
        Path("topic_embeddings.json"),
        Path.cwd() / "K_Means_Drift" / "topic_embeddings.json",
        Path.cwd() / "topic_embeddings.json",
        Path.cwd().parent / "K_Means_Drift" / "topic_embeddings.json"
    ]

    resolved_json = next((p for p in candidate_paths if p.exists()), None)
    if resolved_json is None:
        raise FileNotFoundError(
            "Could not find topic_embeddings.json. Tried: " + ", ".join(str(p) for p in candidate_paths)
        )

    with open(resolved_json, "r", encoding="utf-8") as f:
        payload = json.load(f)

    topic_embeddings = {}
    expected_dim = int(config["embedding_dim"])

    for topic in topics:
        if topic not in payload:
            raise KeyError(f"Missing topic '{topic}' in {resolved_json}")

        vec = np.asarray(payload[topic], dtype=np.float32).flatten()
        if vec.shape[0] != expected_dim:
            raise ValueError(
                f"Embedding dimension mismatch for {topic}: {vec.shape[0]} != {expected_dim}"
            )

        vec = vec / (np.linalg.norm(vec) + 1e-8)
        topic_embeddings[topic] = vec.astype(np.float32)

    print("Loaded topic embeddings from:", resolved_json)
    return topic_embeddings

def soft_topic_label_sentences(sentence_dataframe, topic_embeddings, config):
    rows = []
    topic_names = config["topics"]
    topic_matrix = np.stack([topic_embeddings[name] for name in topic_names]).astype(np.float32)
    threshold = float(config["topic_threshold"])

    if sentence_dataframe.empty:
        return pd.DataFrame(columns=["date", "article_id", "sentence_id", "sentence_text", "sentence_order", "sentence_embeddings", "dominant_topic", "dominant_score"] + topic_names)

    for row in sentence_dataframe.itertuples(index=False):
        emb = np.asarray(row.sentence_embeddings, dtype=np.float32)
        emb = emb / (np.linalg.norm(emb) + 1e-8)

        # Cosine similarities to topic prototypes.
        cosine_scores = np.dot(topic_matrix, emb).astype(np.float32)

        # Convert to a probability-style distribution: non-negative and sums to 1.
        positive_scores = np.clip(cosine_scores, a_min=0.0, a_max=None).astype(np.float32)
        score_sum = float(np.sum(positive_scores))

        if score_sum <= 1e-8:
            topic_probs = np.zeros(len(topic_names), dtype=np.float32)
            topic_probs[int(np.argmax(cosine_scores))] = np.float32(1.0)
        else:
            topic_probs = (positive_scores / score_sum).astype(np.float32)

        dominant_idx = int(np.argmax(topic_probs))
        dominant_score = float(topic_probs[dominant_idx])
        dominant_topic = topic_names[dominant_idx] if dominant_score >= threshold else "Unassigned"

        record = {
            "date": row.date,
            "article_id": row.article_id,
            "sentence_id": row.sentence_id,
            "sentence_text": row.sentence_text,
            "sentence_order": int(row.sentence_order),
            "sentence_embeddings": emb.astype(np.float32),
            "dominant_topic": dominant_topic,
            "dominant_score": dominant_score
        }

        for topic_idx, topic_name in enumerate(topic_names):
            record[topic_name] = np.float32(topic_probs[topic_idx])

        rows.append(record)

    return pd.DataFrame(rows)

def filter_user_topic_sentences(labeled_sentence_dataframe, user_topic, config):
    threshold = float(config["topic_threshold"])

    # Multi-label routing on normalized topic probabilities.
    filtered = labeled_sentence_dataframe[
        labeled_sentence_dataframe[user_topic] >= threshold
    ].copy()
    filtered["selected_topic"] = user_topic
    filtered["similarity_score"] = filtered[user_topic].astype(np.float32)
    filtered["similarity_percent"] = (filtered[user_topic] * 100.0).astype(np.float32)
    return filtered.sort_values(["date", "article_id", "sentence_order"]).reset_index(drop=True)

topic_embeddings = load_topic_embedding_prototypes_from_json(TOPICS, INFERENCE_CONFIG)
labeled_sentence_df = soft_topic_label_sentences(sentence_df, topic_embeddings, INFERENCE_CONFIG)

print("Dominant topic counts (thresholded at", INFERENCE_CONFIG["topic_threshold"], "):")
print(labeled_sentence_df["dominant_topic"].value_counts(dropna=False))

print("\nPer-topic sentence counts at threshold >=", INFERENCE_CONFIG["topic_threshold"], "(>=", int(INFERENCE_CONFIG["topic_threshold"] * 100), "%)")
for topic in TOPICS:
    count_at_threshold = int((labeled_sentence_df[topic] >= float(INFERENCE_CONFIG["topic_threshold"])).sum())
    print(f"{topic}: {count_at_threshold}")

filtered_by_topic = {}
topic_models = {}

for topic in TOPICS:
    filtered_topic_df = filter_user_topic_sentences(labeled_sentence_df, topic, INFERENCE_CONFIG)
    filtered_by_topic[topic] = filtered_topic_df
    print(f"{topic}: filtered sentence count = {len(filtered_topic_df)}")

    if len(filtered_topic_df) < NUM_CLUSTERS:
        print(f"{topic}: skipped model training (rows < {NUM_CLUSTERS})")
        continue

    X_topic = np.vstack(filtered_topic_df["sentence_embeddings"]).astype(np.float32)
    model = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
    model.fit(X_topic)
    topic_models[topic] = model
    print(f"{topic}: model trained")

print("Models trained:", list(topic_models.keys()))

Loaded topic embeddings from: topic_embeddings.json
Dominant topic counts (thresholded at 0.27 ):
dominant_topic
War    28
Name: count, dtype: int64

Per-topic sentence counts at threshold >= 0.27 (>= 27 %)
War: 28
Health: 0
Technology: 0
Climate: 0
Economics: 0
War: filtered sentence count = 28
War: model trained
Health: filtered sentence count = 0
Health: skipped model training (rows < 5)
Technology: filtered sentence count = 0
Technology: skipped model training (rows < 5)
Climate: filtered sentence count = 0
Climate: skipped model training (rows < 5)
Economics: filtered sentence count = 0
Economics: skipped model training (rows < 5)
Models trained: ['War']


## Step 11: Build Topic-wise Inference Matrices
**What this does:** Converts each filtered topic sentence set into an embedding matrix for prediction.
**How it works:** Stacks `sentence_embeddings` using `np.vstack` per topic, skipping empty sets.
**Visual purpose:** Prepares clean per-topic numeric inputs for model inference.

In [52]:
topic_test_matrices = {}

for topic, topic_df in filtered_by_topic.items():
    if topic not in topic_models:
        print(f"{topic}: skipped (no trained model)")
        continue
    if topic_df.empty:
        print(f"{topic}: skipped (no filtered sentences)")
        continue

    X_topic_test = np.vstack(topic_df["sentence_embeddings"]).astype(np.float32)
    topic_test_matrices[topic] = X_topic_test
    print(f"{topic}: test matrix shape = {X_topic_test.shape}")

War: test matrix shape = (28, 768)
Health: skipped (no trained model)
Technology: skipped (no trained model)
Climate: skipped (no trained model)
Economics: skipped (no trained model)


## Step 12: Predict Clusters on Filtered User Sentences
**What this does:** Runs each topic model on its own filtered sentence set.
**How it works:** Predicts cluster labels topic-by-topic and stores per-topic inference DataFrames.
**Visual purpose:** Produces interpretable sentence-level cluster assignments for each topic stream.

In [53]:
inference_by_topic = {}

for topic, model in topic_models.items():
    topic_df = filtered_by_topic.get(topic, pd.DataFrame()).copy()
    X_topic_test = topic_test_matrices.get(topic)

    if X_topic_test is None or topic_df.empty:
        print(f"{topic}: no inference output")
        continue

    topic_df["cluster"] = model.predict(X_topic_test)
    inference_by_topic[topic] = topic_df
    print(f"{topic}: predicted {len(topic_df)} sentence clusters")

preview_frames = []
for topic, topic_df in inference_by_topic.items():
    preview = topic_df[["date", "sentence_text", "cluster"]].head(3).copy()
    preview.insert(0, "topic", topic)
    preview_frames.append(preview)

if preview_frames:
    pd.concat(preview_frames, ignore_index=True)
else:
    pd.DataFrame(columns=["topic", "date", "sentence_text", "cluster"])

War: predicted 28 sentence clusters


## Step 13: Build Article Distributions Per Topic
**What this does:** Creates date-wise cluster distributions separately for each topic model.
**How it works:** Groups by date, computes normalized frequencies for each topic-specific cluster column, and stores results in a dictionary.
**Visual purpose:** Produces comparable topic-wise narrative fingerprints over time.

In [54]:
article_df_by_topic = {}

for topic, topic_df in inference_by_topic.items():
    article_clusters = []

    for date, group in topic_df.groupby("date"):
        counts = group["cluster"].value_counts(normalize=True)
        dist = np.zeros(NUM_CLUSTERS, dtype=np.float32)

        for c, v in counts.items():
            dist[int(c)] = np.float32(v)

        article_clusters.append({
            "date": pd.to_datetime(date),
            "distribution": dist,
            "sentences": group["sentence_text"].tolist()
        })

    article_df = pd.DataFrame(article_clusters).sort_values("date")
    article_df_by_topic[topic] = article_df
    print(f"{topic}: built {len(article_df)} date distributions")

War: built 5 date distributions


## Step 14: Detect and Print Drift for All Topics
**What this does:** Runs drift detection independently for each topic and prints findings.
**How it works:** Computes Jensen-Shannon distance between consecutive dates within each topic-specific article distribution sequence.
**Visual purpose:** Surfaces where narrative changes happen per topic, not just for a single model.

In [55]:
DRIFT_THRESHOLD = 0.3
SENTENCE_PAIR_LIMIT = 2

def _normalize_embeddings(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-8
    return mat / norms

def _build_sentence_change_pairs(before_df, after_df, max_pairs=2):
    if before_df.empty or after_df.empty:
        return []

    before_emb = np.vstack(before_df["sentence_embeddings"].values).astype(np.float32)
    after_emb = np.vstack(after_df["sentence_embeddings"].values).astype(np.float32)

    before_norm = _normalize_embeddings(before_emb)
    after_norm = _normalize_embeddings(after_emb)
    sim_matrix = np.dot(before_norm, after_norm.T)

    used_after = set()
    pairs = []

    for i in range(sim_matrix.shape[0]):
        ranked_after = np.argsort(-sim_matrix[i])
        picked_j = None
        for j in ranked_after:
            j = int(j)
            if j not in used_after:
                picked_j = j
                used_after.add(j)
                break

        if picked_j is None:
            continue

        similarity = float(sim_matrix[i, picked_j])
        change_score = float(1.0 - similarity)
        pairs.append({
            "before_sentence": str(before_df.iloc[i]["sentence_text"]),
            "after_sentence": str(after_df.iloc[picked_j]["sentence_text"]),
            "similarity": similarity,
            "change_score": change_score
        })

    pairs.sort(key=lambda x: x["change_score"], reverse=True)
    return pairs[:max_pairs]

print("\n===== Compact Multi-Topic Narrative Drift Report =====")

topic_drift_summary = {}

for topic in TOPICS:
    article_df = article_df_by_topic.get(topic, pd.DataFrame())
    topic_df = inference_by_topic.get(topic, pd.DataFrame())

    print(f"\n--- Topic: {topic} ---")

    if topic_df.empty or article_df.empty:
        print("No inference output")
        topic_drift_summary[topic] = {
            "num_pairs": 0,
            "num_shifts": 0,
            "max_drift": None,
            "mean_drift": None,
            "all_drifts": []
        }
        continue

    if len(article_df) < 2:
        print("Not enough date points for drift detection")
        topic_drift_summary[topic] = {
            "num_pairs": 0,
            "num_shifts": 0,
            "max_drift": None,
            "mean_drift": None,
            "all_drifts": []
        }
        continue

    drift_rows = []
    drift_scores = []

    for i in range(len(article_df) - 1):
        date_before = pd.to_datetime(article_df.iloc[i]["date"])
        date_after = pd.to_datetime(article_df.iloc[i + 1]["date"])

        p = article_df.iloc[i]["distribution"]
        q = article_df.iloc[i + 1]["distribution"]
        drift = float(jensenshannon(p, q))
        drift_scores.append(drift)

        drift_rows.append({
            "i": i,
            "date_before": date_before,
            "date_after": date_after,
            "drift": drift
        })

    shift_rows = [row for row in drift_rows if row["drift"] > DRIFT_THRESHOLD]
    shift_count = len(shift_rows)

    topic_drift_summary[topic] = {
        "num_pairs": len(drift_scores),
        "num_shifts": shift_count,
        "max_drift": max(drift_scores) if drift_scores else None,
        "mean_drift": (sum(drift_scores) / len(drift_scores)) if drift_scores else None,
        "all_drifts": [round(score, 4) for score in drift_scores]
    }

    if not shift_rows:
        print("No narrative shift above threshold")
        continue

    strongest_shift = max(shift_rows, key=lambda row: row["drift"])
    print("Strongest Shift:")
    print("From:", strongest_shift["date_before"].date())
    print("To:", strongest_shift["date_after"].date())
    print("Drift Score:", strongest_shift["drift"])

    before_sent_df = topic_df[
        pd.to_datetime(topic_df["date"]) == strongest_shift["date_before"]
    ].reset_index(drop=True)
    after_sent_df = topic_df[
        pd.to_datetime(topic_df["date"]) == strongest_shift["date_after"]
    ].reset_index(drop=True)

    sentence_pairs = _build_sentence_change_pairs(
        before_sent_df,
        after_sent_df,
        max_pairs=SENTENCE_PAIR_LIMIT
    )

    if sentence_pairs:
        print("Top sentence changes:")
        for idx, pair in enumerate(sentence_pairs, start=1):
            print(f"Pair {idx} | Change: {pair['change_score']:.4f} | Sim: {pair['similarity']:.4f}")
            print("Before:", pair["before_sentence"])
            print("After :", pair["after_sentence"])
    else:
        print("No sentence-level pairs available for strongest shift")

print("\n===== Topic-wise Drift Summary =====")
for topic in TOPICS:
    stats = topic_drift_summary.get(topic)
    if stats is None:
        print(f"{topic}: no inference output")
        continue

    print(
        f"{topic}: pairs={stats['num_pairs']}, shifts>{DRIFT_THRESHOLD}={stats['num_shifts']}, "
        f"max={stats['max_drift']}, mean={stats['mean_drift']}, drifts={stats['all_drifts']}"
    )


===== Compact Multi-Topic Narrative Drift Report =====

--- Topic: War ---
Strongest Shift:
From: 2022-02-15
To: 2022-02-25
Drift Score: 0.8325546383857727
Top sentence changes:
Pair 1 | Change: 0.5395 | Sim: 0.4605
Before: International observers expressed concern that the growing tensions could lead to a broader conflict in the region.
After : Ukrainian President Volodymyr Zelenskyy declared martial law and called on citizens to defend the country.
Pair 2 | Change: 0.4750 | Sim: 0.5250
Before: Meanwhile, residents in eastern Ukraine reported an increase in ceasefire violations along the contact line separating Ukrainian forces and Russian-backed separatists.
After : Russian troops advanced from several directions, including the north through Belarus and from the eastern region of Donbas.

--- Topic: Health ---
No inference output

--- Topic: Technology ---
No inference output

--- Topic: Climate ---
No inference output

--- Topic: Economics ---
No inference output

===== Topic-wise 

## Step 15: Batch Processing in Correct Order
**Phase 1 (Input Discovery):** Scan the `newinput` root, collect all `_combined.csv` files, validate shape and preview what will be processed.
**Phase 2 (Implementation):** Run the complete topic-wise drift pipeline on each discovered input and save outputs per case.
**Output Goal:** One JSON + one CSV per input case, and explicit `No narrative found` for topics with no detected shift.

In [56]:
from datetime import datetime
import json
from pathlib import Path

# -----------------------------
# Phase 1: Input discovery first
# -----------------------------
PROJECT_ROOT = Path("/home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection")
INPUT_ROOT_CANDIDATES = [
    Path("../newinput"),
    Path("newinput"),
    PROJECT_ROOT / "newinput"
]

resolved_input_root = next((p.resolve() for p in INPUT_ROOT_CANDIDATES if p.exists()), None)
if resolved_input_root is None:
    raise FileNotFoundError(f"Could not locate newinput folder. Tried: {INPUT_ROOT_CANDIDATES}")

INPUT_ROOT = resolved_input_root
OUTPUT_ROOT = PROJECT_ROOT / "Output" / "K_means_drift"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Keep this step self-contained if Step 14 was not executed.
if "DRIFT_THRESHOLD" not in globals():
    DRIFT_THRESHOLD = 0.3


def collect_input_csv_files(input_root: Path):
    csv_files = sorted(
        p for p in input_root.glob("*/*_combined.csv")
        if p.is_file()
    )
    if not csv_files:
        raise FileNotFoundError(f"No *_combined.csv files found under {input_root}")
    return csv_files


input_csv_files = collect_input_csv_files(INPUT_ROOT)
print(f"Resolved input root: {INPUT_ROOT}")
print(f"Discovered input files: {len(input_csv_files)}")
for p in input_csv_files:
    print(" -", p)


# ---------------------------------------
# Phase 2: Implementation after input scan
# ---------------------------------------
def load_user_input_csv(csv_path: Path):
    user_df = pd.read_csv(csv_path)
    user_df = user_df.loc[:, ~user_df.columns.str.contains(r"^Unnamed")].copy()

    required_cols = {"date", "article"}
    missing_cols = required_cols - set(user_df.columns)
    if missing_cols:
        raise ValueError(f"{csv_path.name}: missing required columns {missing_cols}")

    return user_df


def build_topic_models_and_inference(user_df, topic_embeddings, config):
    sentence_df = split_articles_into_sentences(user_df)
    sentence_df = build_context_texts(sentence_df, int(config["context_window"]))
    sentence_df = generate_contextual_sbert_embeddings(sentence_df, config)

    labeled_sentence_df = soft_topic_label_sentences(sentence_df, topic_embeddings, config)

    filtered_by_topic = {}
    topic_models = {}

    for topic in TOPICS:
        filtered_topic_df = filter_user_topic_sentences(labeled_sentence_df, topic, config)
        filtered_by_topic[topic] = filtered_topic_df

        if len(filtered_topic_df) < NUM_CLUSTERS:
            continue

        X_topic = np.vstack(filtered_topic_df["sentence_embeddings"]).astype(np.float32)
        model = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
        model.fit(X_topic)
        topic_models[topic] = model

    inference_by_topic = {}
    for topic, model in topic_models.items():
        topic_df = filtered_by_topic.get(topic, pd.DataFrame()).copy()
        if topic_df.empty:
            continue

        X_topic_test = np.vstack(topic_df["sentence_embeddings"]).astype(np.float32)
        topic_df["cluster"] = model.predict(X_topic_test)
        inference_by_topic[topic] = topic_df

    article_df_by_topic = {}
    for topic, topic_df in inference_by_topic.items():
        article_clusters = []

        for date, group in topic_df.groupby("date"):
            counts = group["cluster"].value_counts(normalize=True)
            dist = np.zeros(NUM_CLUSTERS, dtype=np.float32)
            for c, v in counts.items():
                dist[int(c)] = np.float32(v)

            article_clusters.append({
                "date": pd.to_datetime(date),
                "distribution": dist
            })

        article_df_by_topic[topic] = pd.DataFrame(article_clusters).sort_values("date")

    return inference_by_topic, article_df_by_topic


def _cluster_distribution(cluster_series, num_clusters):
    counts = cluster_series.value_counts(normalize=True)
    dist = np.zeros(num_clusters, dtype=np.float32)
    for c, v in counts.items():
        dist[int(c)] = np.float32(v)
    return dist


def _serialize_shift_sentences(df, changed_clusters):
    if df.empty:
        return []

    subset = df[df["cluster"].isin(changed_clusters)].copy()
    if subset.empty:
        subset = df.copy()

    rows = []
    for row in subset.sort_values(["sentence_order"]).itertuples(index=False):
        rows.append({
            "sentence_id": str(row.sentence_id),
            "cluster": int(row.cluster),
            "sentence_text": str(row.sentence_text)
        })

    return rows


def _extract_shift_sentences(before_df, after_df, cluster_delta_min=0.1):
    if before_df.empty or after_df.empty:
        return {
            "changed_clusters": [],
            "cluster_delta": [0.0] * NUM_CLUSTERS,
            "before_sentences": [],
            "after_sentences": []
        }

    p = _cluster_distribution(before_df["cluster"], NUM_CLUSTERS)
    q = _cluster_distribution(after_df["cluster"], NUM_CLUSTERS)
    delta = (q - p).astype(np.float32)

    changed_clusters = [
        int(idx) for idx, d in enumerate(delta)
        if abs(float(d)) >= float(cluster_delta_min)
    ]

    if not changed_clusters:
        max_idx = int(np.argmax(np.abs(delta)))
        changed_clusters = [max_idx]

    return {
        "changed_clusters": changed_clusters,
        "cluster_delta": [float(x) for x in delta],
        "before_sentences": _serialize_shift_sentences(before_df, changed_clusters),
        "after_sentences": _serialize_shift_sentences(after_df, changed_clusters)
    }


def detect_all_topic_shifts(inference_by_topic, article_df_by_topic, drift_threshold=0.3):
    topic_results = {}

    for topic in TOPICS:
        topic_df = inference_by_topic.get(topic, pd.DataFrame())
        article_df = article_df_by_topic.get(topic, pd.DataFrame())

        if topic_df.empty or article_df.empty:
            topic_results[topic] = {
                "status": "No narrative found",
                "num_pairs": 0,
                "num_shifts": 0,
                "shifts": []
            }
            continue

        if len(article_df) < 2:
            topic_results[topic] = {
                "status": "No narrative found",
                "num_pairs": 0,
                "num_shifts": 0,
                "shifts": []
            }
            continue

        shifts = []
        pair_count = 0

        for i in range(len(article_df) - 1):
            pair_count += 1
            date_before = pd.to_datetime(article_df.iloc[i]["date"])
            date_after = pd.to_datetime(article_df.iloc[i + 1]["date"])

            p = article_df.iloc[i]["distribution"]
            q = article_df.iloc[i + 1]["distribution"]
            drift = float(jensenshannon(p, q))

            if drift <= float(drift_threshold):
                continue

            before_df = topic_df[
                pd.to_datetime(topic_df["date"]) == date_before
            ].reset_index(drop=True)
            after_df = topic_df[
                pd.to_datetime(topic_df["date"]) == date_after
            ].reset_index(drop=True)

            shift_sentences = _extract_shift_sentences(before_df, after_df, cluster_delta_min=0.1)

            shifts.append({
                "date_before": date_before.strftime("%Y-%m-%d"),
                "date_after": date_after.strftime("%Y-%m-%d"),
                "drift_score": drift,
                "changed_clusters": shift_sentences["changed_clusters"],
                "cluster_delta": shift_sentences["cluster_delta"],
                "before_sentences": shift_sentences["before_sentences"],
                "after_sentences": shift_sentences["after_sentences"]
            })

        if shifts:
            topic_results[topic] = {
                "status": "Narrative shifts found",
                "num_pairs": pair_count,
                "num_shifts": len(shifts),
                "shifts": shifts
            }
        else:
            topic_results[topic] = {
                "status": "No narrative found",
                "num_pairs": pair_count,
                "num_shifts": 0,
                "shifts": []
            }

    return topic_results


def save_case_txt_output(input_csv_path: Path, topic_results, output_root: Path):
    out_path = output_root / f"{input_csv_path.stem}.txt"

    lines = []
    lines.append(f"Input File: {input_csv_path}")
    lines.append(f"Generated At: {datetime.now().isoformat(timespec='seconds')}")
    lines.append("")

    for topic in TOPICS:
        topic_data = topic_results.get(topic, {})
        status = topic_data.get("status", "No narrative found")
        shifts = topic_data.get("shifts", [])

        lines.append("=" * 80)
        lines.append(f"Topic: {topic}")
        lines.append(f"Status: {status}")
        lines.append(f"Date Pairs Checked: {topic_data.get('num_pairs', 0)}")
        lines.append(f"Shifts Found: {topic_data.get('num_shifts', 0)}")

        if not shifts:
            lines.append("No narrative found")
            lines.append("")
            continue

        for idx, shift in enumerate(shifts, start=1):
            lines.append("-" * 80)
            lines.append(f"Shift #{idx}")
            lines.append(f"From: {shift['date_before']}")
            lines.append(f"To:   {shift['date_after']}")
            lines.append(f"Drift Score: {shift['drift_score']:.6f}")
            lines.append(f"Changed Clusters: {shift['changed_clusters']}")
            lines.append(f"Cluster Delta: {[round(v, 4) for v in shift['cluster_delta']]}")
            lines.append("Before Sentences:")

            before_sents = shift.get("before_sentences", [])
            if before_sents:
                for sent in before_sents:
                    lines.append(f"  - [{sent['cluster']}] {sent['sentence_text']}")
            else:
                lines.append("  - No narrative found")

            lines.append("After Sentences:")
            after_sents = shift.get("after_sentences", [])
            if after_sents:
                for sent in after_sents:
                    lines.append(f"  - [{sent['cluster']}] {sent['sentence_text']}")
            else:
                lines.append("  - No narrative found")

            lines.append("")

    out_path.write_text("\n".join(lines), encoding="utf-8")
    return out_path


# Execute batch after input file discovery.
topic_embeddings_batch = load_topic_embedding_prototypes_from_json(TOPICS, INFERENCE_CONFIG)

batch_summary = []
for csv_path in input_csv_files:
    case_name = csv_path.parent.name
    print(f"\nProcessing: {case_name}")

    try:
        user_df = load_user_input_csv(csv_path)
        inference_by_topic, article_df_by_topic = build_topic_models_and_inference(
            user_df,
            topic_embeddings_batch,
            INFERENCE_CONFIG
        )
        topic_results = detect_all_topic_shifts(
            inference_by_topic,
            article_df_by_topic,
            drift_threshold=DRIFT_THRESHOLD
        )

        txt_out = save_case_txt_output(csv_path, topic_results, OUTPUT_ROOT)
        total_shifts = sum(topic_results[t]["num_shifts"] for t in TOPICS)

        batch_summary.append({
            "case_name": case_name,
            "input_file": str(csv_path),
            "total_shifts": int(total_shifts),
            "txt_output": str(txt_out),
            "status": "ok"
        })

        print(f"Saved: {txt_out.name} | total shifts: {total_shifts}")

    except Exception as exc:
        batch_summary.append({
            "case_name": case_name,
            "input_file": str(csv_path),
            "total_shifts": 0,
            "txt_output": "",
            "status": f"error: {exc}"
        })
        print(f"Failed: {case_name} | {exc}")

summary_df = pd.DataFrame(batch_summary)
summary_path = OUTPUT_ROOT / "batch_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nBatch complete")
print(f"Summary saved: {summary_path}")
summary_df

Resolved input root: /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput
Discovered input files: 15
 - /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput/Ner1_hard/Ner1_hard_combined.csv
 - /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput/Ner1_low/Ner1_low_combined.csv
 - /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput/Ner1_medium/Ner1_medium_combined.csv
 - /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput/Ner2_hard/Ner2_hard_combined.csv
 - /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput/Ner2_low/Ner2_low_combined.csv
 - /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput/Ner2_medium/Ner2_medium_combined.csv
 - /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/newinput/N

,case_name,input_file,total_shifts,txt_output,status
0,Ner1_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner1_hard_combined.csv: missing require...
1,Ner1_low,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner1_low_combined.csv: missing required...
2,Ner1_medium,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner1_medium_combined.csv: missing requi...
3,Ner2_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner2_hard_combined.csv: missing require...
4,Ner2_low,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner2_low_combined.csv: missing required...
5,Ner2_medium,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner2_medium_combined.csv: missing requi...
6,Ner3_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner3_hard_combined.csv: missing require...
7,Ner3_low,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner3_low_combined.csv: missing required...
8,Ner3_medium,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner3_medium_combined.csv: missing requi...
9,Ner4_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,0,,error: Ner4_hard_combined.csv: missing require...


In [57]:
# Output path override: keep all batch artifacts in Output/K_means_drift.
OUTPUT_ROOT = PROJECT_ROOT / "Output" / "K_means_drift"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output root set to:", OUTPUT_ROOT)

Output root set to: /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/Output/K_means_drift


## Step 16: Header Normalization Fix + Re-run Batch TXT Export
This step normalizes `Date`/`Article Text` style headers to `date`/`article` and re-runs the per-file TXT export for all discovered inputs.

In [58]:
def load_user_input_csv(csv_path: Path):
    user_df = pd.read_csv(csv_path)
    user_df = user_df.loc[:, ~user_df.columns.str.contains(r"^Unnamed")].copy()

    normalized = {str(c).strip().lower(): c for c in user_df.columns}

    date_col = None
    article_col = None

    for cand in ["date", "dates", "published_date", "published date"]:
        if cand in normalized:
            date_col = normalized[cand]
            break

    for cand in ["article", "article text", "text", "content", "article_text"]:
        if cand in normalized:
            article_col = normalized[cand]
            break

    if date_col is None or article_col is None:
        raise ValueError(
            f"{csv_path.name}: expected date/article-like columns, found {list(user_df.columns)}"
        )

    user_df = user_df.rename(columns={date_col: "date", article_col: "article"})
    return user_df[["date", "article"]].copy()


batch_summary = []
for csv_path in input_csv_files:
    case_name = csv_path.parent.name
    print(f"\nProcessing (normalized): {case_name}")

    try:
        user_df = load_user_input_csv(csv_path)
        inference_by_topic, article_df_by_topic = build_topic_models_and_inference(
            user_df,
            topic_embeddings_batch,
            INFERENCE_CONFIG
        )
        topic_results = detect_all_topic_shifts(
            inference_by_topic,
            article_df_by_topic,
            drift_threshold=DRIFT_THRESHOLD
        )

        txt_out = save_case_txt_output(csv_path, topic_results, OUTPUT_ROOT)
        total_shifts = sum(topic_results[t]["num_shifts"] for t in TOPICS)

        batch_summary.append({
            "case_name": case_name,
            "input_file": str(csv_path),
            "total_shifts": int(total_shifts),
            "txt_output": str(txt_out),
            "status": "ok"
        })

        print(f"Saved: {txt_out.name} | total shifts: {total_shifts}")

    except Exception as exc:
        batch_summary.append({
            "case_name": case_name,
            "input_file": str(csv_path),
            "total_shifts": 0,
            "txt_output": "",
            "status": f"error: {exc}"
        })
        print(f"Failed: {case_name} | {exc}")

summary_df = pd.DataFrame(batch_summary)
summary_path = OUTPUT_ROOT / "batch_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nRe-run complete")
print(f"Summary saved: {summary_path}")
summary_df


Processing (normalized): Ner1_hard


/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4120.96it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: Ner1_hard_combined.txt | total shifts: 17

Processing (normalized): Ner1_low


/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3923.78it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: Ner1_low_combined.txt | total shifts: 8

Processing (normalized): Ner1_medium


...

Saved: Ner4_low_combined.txt | total shifts: 22

Processing (normalized): Ner4_medium


/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3609.60it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: Ner4_medium_combined.txt | total shifts: 14

Processing (normalized): Ner5_hard


/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3641.30it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

## Step 17: Stable Local SBERT Cache + Final Re-run
This step caches the SBERT model once in memory and forces local loading to avoid repeated remote checks during batch processing.

In [ ]:
SBERT_MODEL_CACHE = None


def generate_contextual_sbert_embeddings(sentence_dataframe, config, sbert_model_name="all-mpnet-base-v2"):
    global SBERT_MODEL_CACHE

    if sentence_dataframe.empty:
        sentence_dataframe["sentence_embeddings"] = []
        return sentence_dataframe

    if int(config["embedding_dim"]) != 768:
        raise ValueError("Inference requires config['embedding_dim'] == 768")

    if SBERT_MODEL_CACHE is None:
        SBERT_MODEL_CACHE = SentenceTransformer(
            sbert_model_name,
            device="cpu",
            local_files_only=True
        )

    encoded = SBERT_MODEL_CACHE.encode(
        sentence_dataframe["context_text"].tolist(),
        batch_size=int(config["inference_batch_size"]),
        show_progress_bar=False,
        convert_to_numpy=True
    )
    encoded = np.asarray(encoded, dtype=np.float32)

    if encoded.shape[1] != config["embedding_dim"]:
        raise ValueError(
            f"SBERT output dim {encoded.shape[1]} does not match config['embedding_dim']={config['embedding_dim']}"
        )

    sentence_dataframe = sentence_dataframe.copy()
    sentence_dataframe["sentence_embeddings"] = [vec.astype(np.float32) for vec in encoded]
    return sentence_dataframe


# Re-run batch after stabilizing model loading.
batch_summary = []
for csv_path in input_csv_files:
    case_name = csv_path.parent.name
    print(f"\nProcessing (final): {case_name}")

    try:
        user_df = load_user_input_csv(csv_path)
        inference_by_topic, article_df_by_topic = build_topic_models_and_inference(
            user_df,
            topic_embeddings_batch,
            INFERENCE_CONFIG
        )
        topic_results = detect_all_topic_shifts(
            inference_by_topic,
            article_df_by_topic,
            drift_threshold=DRIFT_THRESHOLD
        )

        txt_out = save_case_txt_output(csv_path, topic_results, OUTPUT_ROOT)
        total_shifts = sum(topic_results[t]["num_shifts"] for t in TOPICS)

        batch_summary.append({
            "case_name": case_name,
            "input_file": str(csv_path),
            "total_shifts": int(total_shifts),
            "txt_output": str(txt_out),
            "status": "ok"
        })

        print(f"Saved: {txt_out.name} | total shifts: {total_shifts}")

    except Exception as exc:
        batch_summary.append({
            "case_name": case_name,
            "input_file": str(csv_path),
            "total_shifts": 0,
            "txt_output": "",
            "status": f"error: {exc}"
        })
        print(f"Failed: {case_name} | {exc}")

summary_df = pd.DataFrame(batch_summary)
summary_path = OUTPUT_ROOT / "batch_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nFinal batch complete")
print(f"Summary saved: {summary_path}")
summary_df

/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)



Processing (final): Ner1_hard


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2840.92it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: Ner1_hard_combined.txt | total shifts: 17

Processing (final): Ner1_low


/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)


...

/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)


Saved: Ner5_low_combined.txt | total shifts: 5

Processing (final): Ner5_medium


/tmp/ipykernel_201500/1764941769.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date_value = pd.to_datetime(row["date"], errors="coerce", dayfirst=True)


Saved: Ner5_medium_combined.txt | total shifts: 9

Final batch complete
Summary saved: /home/param-modi/myFolder/IIIT /Semester 2/CS7.401 INLP/Narrative-Shift-Detection/Output/All_15_Narrative_Shifts_TXT/batch_summary.csv


,case_name,input_file,total_shifts,txt_output,status
0,Ner1_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,17,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
1,Ner1_low,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,8,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
2,Ner1_medium,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,11,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
3,Ner2_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,5,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
4,Ner2_low,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,17,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
5,Ner2_medium,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,17,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
6,Ner3_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,8,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
7,Ner3_low,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,4,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
8,Ner3_medium,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,5,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok
9,Ner4_hard,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,19,/home/param-modi/myFolder/IIIT /Semester 2/CS7...,ok


## Step 18: Case-Insensitive Column Alias Update
Accept `date` in any case and article aliases: `article`, `articles`, `article's` (plus existing text variants).

In [ ]:
def load_user_input_csv(csv_path: Path):
    user_df = pd.read_csv(csv_path)
    user_df = user_df.loc[:, ~user_df.columns.str.contains(r"^Unnamed")].copy()

    # Case-insensitive column matching with article aliases.
    normalized = {str(c).strip().lower(): c for c in user_df.columns}

    date_col = None
    article_col = None

    for cand in ["date", "dates", "published_date", "published date"]:
        if cand in normalized:
            date_col = normalized[cand]
            break

    for cand in [
        "article",
        "articles",
        "article's",
        "article text",
        "article_text",
        "text",
        "content",
    ]:
        if cand in normalized:
            article_col = normalized[cand]
            break

    if date_col is None or article_col is None:
        raise ValueError(
            f"{csv_path.name}: expected date/article-like columns, found {list(user_df.columns)}"
        )

    user_df = user_df.rename(columns={date_col: "date", article_col: "article"})
    return user_df[["date", "article"]].copy()

print("Updated loader active: case-insensitive 'date' and article aliases accepted.")

## Step 19: Silence Date Warning + Keep Single SBERT Load
This step updates date parsing to avoid the `dayfirst=True` warning on ISO dates and keeps SBERT cached so model weights are loaded once per batch run.

In [ ]:
# Drop-in override for cleaner logs and deterministic date parsing.
# Run this cell once, then rerun the final batch cell.

def split_articles_into_sentences(input_dataframe):
    import re

    sentence_rows = []
    required_cols = {"date", "article"}
    missing = required_cols - set(input_dataframe.columns)
    if missing:
        raise ValueError(f"Input CSV must contain columns: {required_cols}. Missing: {missing}")

    for article_idx, row in input_dataframe.reset_index(drop=True).iterrows():
        article_id = row.get("article_id", f"article_{article_idx}")

        # Parse ISO-like dates first (YYYY-MM-DD), then fallback to day-first parsing.
        date_raw = str(row["date"]).strip()
        date_value = pd.to_datetime(date_raw, errors="coerce", format="%Y-%m-%d")
        if pd.isna(date_value):
            date_value = pd.to_datetime(date_raw, errors="coerce", dayfirst=True)
        if pd.isna(date_value):
            continue

        text = str(row["article"]).strip()
        if not text or text.lower() == "nan":
            continue

        sentence_list = [
            s.strip()
            for s in re.split(r"(?<=[.!?])\s+", text)
            if s and s.strip()
        ]

        for sentence_order, sentence_text in enumerate(sentence_list):
            sentence_id = f"{article_id}_s{sentence_order}"
            sentence_rows.append({
                "date": date_value.normalize(),
                "article_id": str(article_id),
                "sentence_id": sentence_id,
                "sentence_text": sentence_text,
                "sentence_order": int(sentence_order)
            })

    sentence_dataframe = pd.DataFrame(sentence_rows)
    if sentence_dataframe.empty:
        return pd.DataFrame(columns=["date", "article_id", "sentence_id", "sentence_text", "sentence_order"])
    return sentence_dataframe[["date", "article_id", "sentence_id", "sentence_text", "sentence_order"]]


# Ensure SBERT cache exists and is reused across all files in the batch.
if "SBERT_MODEL_CACHE" not in globals():
    SBERT_MODEL_CACHE = None

print("Step 19 override active: ISO dates parsed without warning; SBERT cache preserved.")